In [2]:
import pandas as pd
from datetime import datetime
from io import BytesIO
from google.cloud import storage, bigquery
import numpy as np
import warnings
pd.set_option('display.max_columns', None)
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

In [2]:
schema_Bases = [
        bigquery.SchemaField("CERTIFICADO_BANCO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CODIGO_PRODUCTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("PRODUCTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_ALTA", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("FECHA_BAJA", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("DIFERENCIA_DIAS", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("MONEDA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("PRIMA", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("GLOSA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NUMERO_OPERACION", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_OPERACION", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("ORIGEN", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CUENTA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FUENTE", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NUMERO_RECLAMO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_CIERRE", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("NOMBRE_ARCHIVO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("TIPO_BASE", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_CARGA", bigquery.enums.SqlTypeNames.DATE)
]

In [4]:
df_RECLAMOS= pd.read_excel("C:/data/BASES CONCILIACIONES_13-04-26.xlsx", sheet_name=0, dtype=str)
df_RECLAMOS.columns = (df_RECLAMOS.columns.str.strip().str.upper().str.replace(r'[^A-Za-z0-9]', '_', regex=True))

In [14]:
df_RECLAMOS= df_RECLAMOS.rename(columns={'CERTIFICADO':'CERTIFICADO_BANCO', 
                               'IMPORTE_A_DEVOLVER':'PRIMA', 
                               'FECHA_DE_ENVIO_A_OPERACIONES':'FECHA_OPERACION'})

In [19]:
df_RECLAMOS['MONEDA'].value_counts()

MONEDA
PEN    46911
USD    28387
Name: count, dtype: int64

In [16]:
df_RECLAMOS['PRODUCTO'] = df_RECLAMOS['PRODUCTO'].str.upper()
#df_RECLAMOS['PRIMA'] = df_RECLAMOS['PRIMA'].replace({',': ''}, regex=True)
df_RECLAMOS['PRIMA'] = pd.to_numeric(df_RECLAMOS['PRIMA'], errors="coerce").astype('float64')
df_RECLAMOS['FECHA_OPERACION'] = pd.to_datetime(df_RECLAMOS['FECHA_OPERACION'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
df_RECLAMOS.loc[df_RECLAMOS['MONEDA'].fillna('').str.strip().str.lower().isin(['soles','sol']), 'MONEDA'] = 'PEN'
df_RECLAMOS.loc[df_RECLAMOS['MONEDA'].fillna('').str.strip().str.lower().isin(['dólares', 'dolares']), 'MONEDA'] = 'USD'

In [17]:
df_RECLAMOS= df_RECLAMOS[['CERTIFICADO_BANCO','PRODUCTO','MONEDA','PRIMA','FECHA_OPERACION']]

In [5]:
df_RECLAMOS.head(3)

,FECHA_DE_ENVIO_A_OPERACIONES,FECHA_DE_ANULACI_N,CERTIFICADO,PRODUCTO,IMPORTE_A_DEVOLVER,MONEDA
0,2024-01-01 00:00:00,NaN,00110178184000382668,SALUD A TU ALCANCE,1138,Soles
1,2024-01-02 00:00:00,NaN,00110183164000988164,PROTECCIÓN DE TARJETA,41,Dólares
2,2024-01-02 00:00:00,NaN,00110194874000475663,MULTIRIESGO NEGOCIO,2294,Soles
